# Tree SHAP + Physical-Feature Dependence Plots on Colab (issue #14)

Runs Tree SHAP analysis for the production Model B-Full model against the 514-sample Testset: computes exact per-feature Tree SHAP contributions over the full 8,964-feature input matrix (8,960 NT-embedding dims + 4 physical features), separates the top-20 sequence-embedding features from the 4 named physical features (MFE, delta-G, Tm, GC), renders a feature-value-vs-SHAP-value dependence plot for each physical feature, and computes the aggregate physical SHAP contribution ratio `sum(|SHAP_physical|) / sum(|SHAP_all|)`.

This runs against the already-trained production `hybrid_xgb_model.json` (no retraining, unlike the ablation study in issue #13) - the only reason it needs Colab rather than a local machine is building NT embeddings for the 514-sequence Testset, the same bottleneck the ISM sweep (issue #11) and mismatch profiling (issue #12) hit.

**One-time setup**: upload these files/folders to a folder in your Google Drive: `NT_sacas9_fintuned_model/` (the whole checkpoint directory), `hybrid_xgb_model.json`, `test_metadata.csv`. Set `DRIVE_DIR` below to that folder's path.

**Output**: `shap_results.json` (top-20 embedding features, all 4 physical features' SHAP, the aggregate physical contribution ratio, and dependence-plot paths), `shap_values.npz` (the full `(514, 8964)` SHAP value matrix), and `shap_dependence_plots/dependence_{MFE,delta_G,Tm,GC}.png` - all downloaded automatically at the end and also kept on Drive as a backup. Copy them back into your local `xai_test/` dir; they have the same schema/contents `step4_shap.py` produces locally.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Point at your uploaded model assets

Edit `DRIVE_DIR` to the Drive folder holding `NT_sacas9_fintuned_model/`, `hybrid_xgb_model.json`, `test_metadata.csv`.

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/xai_test"  # <-- EDIT ME

required = [
    "NT_sacas9_fintuned_model",
    "hybrid_xgb_model.json",
    "test_metadata.csv",
]
missing = [
    name for name in required if not os.path.exists(os.path.join(DRIVE_DIR, name))
]
if missing:
    raise FileNotFoundError(
        f"Missing in {DRIVE_DIR}: {missing}. Upload these to Drive first, or fix DRIVE_DIR."
    )

os.chdir(DRIVE_DIR)
print("Working directory:", os.getcwd())

## 3. Install ViennaRNA CLI (RNAfold / RNAduplex) and Python deps

If `apt-get install vienna-rna` fails (package not found on this Colab image), say so and I'll add a conda-forge/bioconda fallback - don't silently skip this step, `Model_B_Predictor` will fail without both binaries on `PATH`.

In [ ]:
!apt-get -qq update && apt-get -qq install -y vienna-rna
!which RNAfold && which RNAduplex

In [ ]:
!pip install -q transformers biopython xgboost

## 4. Write the predictor + Tree SHAP modules

Mirrors the current committed `xai_test/` repo exactly (embedded at notebook-build time, not fetched at runtime).

In [ ]:
%%writefile model_b_wrapper.py
import os
import pickle
import re
import subprocess

import joblib
import numpy as np
import torch
import xgboost as xgb
from Bio.SeqUtils import MeltingTemp as mt
from Bio.SeqUtils import gc_fraction
from transformers import AutoModelForMaskedLM, AutoTokenizer


class Model_B_Predictor:
    def __init__(self, nt_model_dir: str, xgb_model_path: str):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading NT backbone from {nt_model_dir}...")
        self.tokenizer = AutoTokenizer.from_pretrained(nt_model_dir)
        self.nt_model = AutoModelForMaskedLM.from_pretrained(nt_model_dir).to(
            self.device
        )
        self.nt_model.eval()

        print(f"Loading XGBoost model from {xgb_model_path}...")
        self.xgb_model = self._load_xgb_model(xgb_model_path)

    def _load_xgb_model(self, path: str):
        if not os.path.exists(path):
            raise FileNotFoundError(f"XGBoost model file not found: {path}")

        try:
            with open(path, "rb") as f:
                return pickle.load(f)
        except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
            try:
                return joblib.load(path)
            except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
                model = xgb.XGBRegressor()
                model.load_model(path)
                return model

    def extract_nt_embeddings(self, sequences: list[str]) -> np.ndarray:
        inputs = self.tokenizer(
            sequences, return_tensors="pt", padding=True, truncation=True
        ).to(self.device)
        with torch.no_grad():
            outputs = self.nt_model(**inputs, output_hidden_states=True)
            last_hidden_state = outputs.hidden_states[-1]

            batch_size, seq_len, hidden_dim = last_hidden_state.shape
            assert seq_len == 7, f"Expected 7 tokens, got {seq_len}"
            assert hidden_dim == 1280, f"Expected 1280 hidden dim, got {hidden_dim}"

            flattened_embeddings = last_hidden_state.view(batch_size, -1).cpu().numpy()
        return flattened_embeddings

    def _get_guide_rna(self, dna_seq: str) -> str:
        complement_map = str.maketrans("ATCGatcg", "UAGCuagc")
        return dna_seq.translate(complement_map)[::-1]

    # RNAfold/RNAduplex are external processes: each subprocess.run() call pays
    # OS process-spawn overhead regardless of how little work it does. Both CLIs
    # accept many records over a single stdin stream, so batching sequences into
    # one call (chunked, to keep any single call's stdin/stdout bounded) turns an
    # O(n) process-spawn cost into O(n / batch_size) - the dominant cost at
    # ISM-sweep scale (tens of thousands of mutants). Deliberate scope override of
    # issue #9's "RNAfold/RNAduplex call batching" Out-of-Scope note, requested
    # directly by the user; verified to reproduce the exact same per-sequence
    # energy values as the prior one-process-per-sequence implementation.
    PHYSICAL_FEATURE_BATCH_SIZE = 500

    @staticmethod
    def _chunked(items: list, size: int):
        for i in range(0, len(items), size):
            yield items[i : i + size]

    def _calc_mfe_batch(self, seqs: list[str]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(seqs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAfold CLI 호출 (--noPS: 이미지 생성 방지)
            proc = subprocess.run(
                ["RNAfold", "--noPS"],
                input="".join(f"{seq}\n" for seq in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # Each record prints "<sequence>\n<dot-bracket structure> (<energy>)\n".
            # The structure diagram's own parentheses never have a digit directly
            # inside them, so this regex matches exactly one energy per record,
            # in input order. 출력 예시: ... ( -4.20)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAfold batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def _calc_duplex_dg_batch(self, pairs: list[tuple[str, str]]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(pairs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAduplex CLI 호출
            proc = subprocess.run(
                ["RNAduplex"],
                input="".join(f"{guide}\n{target}\n" for guide, target in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # 출력 예시: ... :  1,36 :   1,36 (-32.50)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAduplex batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def compute_physical_features(self, sequences: list[str]) -> np.ndarray:
        # 1. MFE (RNAfold.exe), batched across all sequences
        mfe_values = self._calc_mfe_batch(sequences)

        # 2. Duplex ΔG (RNAduplex.exe), batched across all sequences
        guide_rnas = [self._get_guide_rna(seq) for seq in sequences]
        dg_values = self._calc_duplex_dg_batch(list(zip(guide_rnas, sequences)))

        features = []
        for seq, mfe, dg in zip(sequences, mfe_values, dg_values):
            # 3. Tm (must match training-time nn_table=DNA_NN4 in nt_feature_stacking.py)
            tm = mt.Tm_NN(seq, nn_table=mt.DNA_NN4)

            # 4. GC Content
            gc = gc_fraction(seq) * 100.0

            features.append([mfe, dg, tm, gc])
        return np.array(features, dtype=np.float32)

    def predict(self, sequences: list[str]) -> np.ndarray:
        embeddings = self.extract_nt_embeddings(sequences)
        physical_feats = self.compute_physical_features(sequences)

        final_features = np.hstack([embeddings, physical_feats])
        assert final_features.shape[1] == 8964, (
            f"Dimension mismatch: Expected 8964, got {final_features.shape[1]}"
        )

        preds = self.xgb_model.predict(final_features)
        return np.array(preds).flatten()


In [ ]:
%%writefile ism_sweep.py
"""In-silico saturation mutagenesis (ISM) sweep (Step 3, see CONTEXT.md / plan_realize.md).

Generates every position x alternative-base substitution for a set of 36bp
Testset sequences, scores wild-type and mutants through a predictor's own
`predict()`, and reduces the result to a relative delta-score sensitivity
array. Works against any predictor exposing `predict(sequences) -> array`
(duck-typed to `Model_A_Predictor` / `Model_B_Predictor`), so Model B's
per-mutant physical-feature recomputation (MFE, dG, Tm, GC) happens for free:
every mutant is dispatched as a full sequence, never a cached wild-type value
or a bare delta.
"""

from collections.abc import Sequence

import numpy as np

BASES = ("A", "C", "G", "T")

# Relative delta-Score is undefined (unbounded) as Score_WT -> 0; scores at or
# below this magnitude are masked to NaN instead of producing a spurious ratio.
RELATIVE_DELTA_NAN_THRESHOLD = 1e-4

# Neither Model_A_Predictor nor Model_B_Predictor batches internally - predict()
# runs whatever list it's given as one forward pass. An ISM sweep can hand it
# tens of thousands of mutants at once, which is large enough to exhaust GPU
# memory (observed: CUDA OOM inside Model A's LSTM forward pass on a 14.56GB
# T4 trying to allocate 20.71 GiB for one such call). Chunking dispatch here
# keeps peak memory bounded regardless of sweep size, without changing either
# predictor's own predict() contract.
DEFAULT_PREDICT_BATCH_SIZE = 256


def alt_bases_for(wt_base: str) -> list[str]:
    """The 3 non-wild-type bases, in fixed lexicographic order."""
    wt_base = wt_base.upper()
    if wt_base not in BASES:
        raise ValueError(f"Not a valid base: {wt_base!r}")
    return sorted(set(BASES) - {wt_base})


def generate_ism_mutants(sequence: str) -> list[str]:
    """All position x alternative-base substitutions for `sequence`.

    Ordered position-major (0..len-1), alternative-base lexicographic-minor,
    matching the `ism_delta` array's (position, alternative_base) axes.
    """
    sequence = sequence.upper()
    mutants = []
    for pos, wt_base in enumerate(sequence):
        for alt_base in alt_bases_for(wt_base):
            mutants.append(sequence[:pos] + alt_base + sequence[pos + 1 :])
    return mutants


def dispatch_in_batches(
    fn, items: list[str], batch_size: int, stack=np.concatenate
) -> np.ndarray:
    """Call `fn()` in bounded chunks and combine the results with `stack`.

    Generic version of the chunking `predict_in_batches` needs for 1D score
    arrays; `step4_ablation.py` reuses this directly (with `stack=np.vstack`)
    to batch `Model_B_Predictor.extract_nt_embeddings()`'s 2D output the same
    way, instead of re-deriving the chunk loop.
    """
    if not items:
        return np.array([])
    chunks = [fn(items[i : i + batch_size]) for i in range(0, len(items), batch_size)]
    return stack(chunks)


def predict_in_batches(predictor, sequences: list[str], batch_size: int) -> np.ndarray:
    """Dispatch `predictor.predict()` in bounded chunks (shared with mismatch_profiling.py)."""
    if not sequences:
        return np.array([], dtype=np.float64)

    def _predict_chunk(chunk):
        return np.asarray(predictor.predict(chunk), dtype=np.float64)

    return dispatch_in_batches(
        _predict_chunk, sequences, batch_size, stack=np.concatenate
    )


def relative_delta(
    wt_scores: np.ndarray, mutant_scores: np.ndarray, nan_threshold: float
) -> np.ndarray:
    """`(mutant - wt) / |wt|` per sample, NaN-masked wherever `|wt| <= nan_threshold`.

    `wt_scores` is 1D (one value per sample); `mutant_scores` broadcasts against
    it along its leading axis - shape `(n, positions, alt_bases)` for the ISM
    sweep, `(n, scenarios)` for complex mismatch profiling. Shared so both
    modules apply the exact same guardrail (see `RELATIVE_DELTA_NAN_THRESHOLD`).
    """
    broadcast_shape = (wt_scores.shape[0],) + (1,) * (mutant_scores.ndim - 1)
    wt_broadcast = wt_scores.reshape(broadcast_shape)
    with np.errstate(divide="ignore", invalid="ignore"):
        delta = (mutant_scores - wt_broadcast) / np.abs(wt_broadcast)
    delta[np.abs(wt_scores) <= nan_threshold] = np.nan
    return delta


def compute_ism_delta(
    sequences: Sequence[str],
    predictor,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> np.ndarray:
    """Run the full ISM sweep for one predictor.

    Returns `ism_delta`, shape `(len(sequences), seq_len, 3)`: the relative
    score delta `(Score_mutant - Score_WT) / |Score_WT|` per position x
    lexicographic alternative base, NaN-masked wherever `|Score_WT| <=
    nan_threshold`. `predict_batch_size` bounds how many sequences reach
    `predictor.predict()` per call; lower it if you still hit an
    out-of-memory error.
    """
    sequences = list(sequences)
    seq_len = len(sequences[0])
    if any(len(seq) != seq_len for seq in sequences):
        raise ValueError("All sequences must share the same length for the ISM sweep.")

    wt_scores = predict_in_batches(predictor, sequences, predict_batch_size)

    mutants: list[str] = []
    for sequence in sequences:
        mutants.extend(generate_ism_mutants(sequence))

    mutant_scores = predict_in_batches(predictor, mutants, predict_batch_size)
    mutant_scores = mutant_scores.reshape(len(sequences), seq_len, 3)

    return relative_delta(wt_scores, mutant_scores, nan_threshold)


def run_ism_sweep(
    sequences: Sequence[str],
    predictor_a,
    predictor_b,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> dict[str, np.ndarray]:
    """Run the ISM sweep for both models over the same Testset sequences."""
    return {
        "ism_delta_model_a": compute_ism_delta(
            sequences, predictor_a, predict_batch_size=predict_batch_size
        ),
        "ism_delta_model_b": compute_ism_delta(
            sequences, predictor_b, predict_batch_size=predict_batch_size
        ),
    }


In [ ]:
%%writefile shap_analysis.py
"""Tree SHAP analysis for production Model B-Full (Step 4-2, issue #14).

Computes exact per-feature Tree SHAP contributions via XGBoost's native
`pred_contribs` prediction mode - the same Tree SHAP algorithm the `shap`
package implements, built directly into the booster, so no extra dependency
is needed. Separates the top-20 sequence-embedding features from the 4 named
physical features (MFE, delta-G, Tm, GC) and reports an aggregate physical
SHAP contribution ratio, matching issue #14's acceptance criteria.
"""

import os

import numpy as np
import xgboost as xgb

EMBEDDING_DIM = 8960
PHYSICAL_DIM = 4
# Column order must match Model_B_Predictor.compute_physical_features's
# hstack order (mfe, dg, tm, gc) - see model_b_wrapper.py.
PHYSICAL_FEATURE_NAMES = ("MFE", "delta_G", "Tm", "GC")

DEFAULT_TOP_K = 20


def compute_tree_shap(model: xgb.XGBRegressor, X: np.ndarray) -> np.ndarray:
    """Exact per-feature Tree SHAP contributions for `X`, bias column dropped.

    `pred_contribs=True` returns one extra trailing column (the expected-value
    bias term) needed for additivity - `row.sum() + bias == raw prediction`.
    That bias isn't a feature contribution, so it's sliced off here; every
    downstream consumer works with a plain `(n_samples, n_features)` matrix.
    """
    booster = model.get_booster()
    dmatrix = xgb.DMatrix(X)
    contribs = booster.predict(dmatrix, pred_contribs=True)
    return contribs[:, :-1]


def split_shap_by_group(
    shap_values: np.ndarray,
    embedding_dim: int = EMBEDDING_DIM,
    physical_dim: int = PHYSICAL_DIM,
) -> dict[str, np.ndarray]:
    """Slice a `(n, embedding_dim + physical_dim)` SHAP matrix by feature group."""
    assert shap_values.shape[1] == embedding_dim + physical_dim, (
        f"Expected {embedding_dim + physical_dim} SHAP columns, got {shap_values.shape[1]}"
    )
    return {
        "embedding": shap_values[:, :embedding_dim],
        "physical": shap_values[:, embedding_dim:],
    }


def top_k_embedding_features(
    shap_embedding: np.ndarray, k: int = DEFAULT_TOP_K
) -> list[dict]:
    """Rank embedding features by mean |SHAP| across samples, descending.

    `feature_index` is local to the embedding sub-block (0..embedding_dim-1,
    e.g. 0-8959 in production), not the full 8,964-wide matrix - callers
    that need a global column index must add `embedding_dim` themselves.
    Caps at the number of available columns rather than raising when `k`
    exceeds the embedding width (relevant for tests against synthetic,
    lower-dimensional embeddings).
    """
    mean_abs = np.abs(shap_embedding).mean(axis=0)
    order = np.argsort(mean_abs)[::-1][:k]
    return [
        {"feature_index": int(i), "mean_abs_shap": float(mean_abs[i])} for i in order
    ]


def physical_feature_shap_summary(shap_physical: np.ndarray) -> list[dict]:
    """Mean |SHAP| per named physical feature, in `PHYSICAL_FEATURE_NAMES` order."""
    mean_abs = np.abs(shap_physical).mean(axis=0)
    return [
        {"feature_name": name, "mean_abs_shap": float(mean_abs[i])}
        for i, name in enumerate(PHYSICAL_FEATURE_NAMES)
    ]


def aggregate_physical_contribution_ratio(
    shap_values: np.ndarray, embedding_dim: int = EMBEDDING_DIM
) -> float:
    """`sum(|SHAP_physical|) / sum(|SHAP_all|)` - a proportion in [0, 1]."""
    total_abs = np.abs(shap_values).sum()
    physical_abs = np.abs(shap_values[:, embedding_dim:]).sum()
    return float(physical_abs / total_abs)


def dependence_plot_data(
    X_physical: np.ndarray,
    shap_physical: np.ndarray,
) -> dict[str, dict[str, np.ndarray]]:
    """Per physical feature: its raw values paired with their own SHAP column."""
    return {
        name: {
            "feature_value": X_physical[:, i],
            "shap_value": shap_physical[:, i],
        }
        for i, name in enumerate(PHYSICAL_FEATURE_NAMES)
    }


def save_dependence_plots(
    X_physical: np.ndarray,
    shap_physical: np.ndarray,
    output_dir: str,
) -> dict[str, str]:
    """Render one feature-value-vs-SHAP-value scatter plot per physical feature."""
    # Imported lazily so importing this module (e.g. for the pure array logic
    # above, as every test in test_shap_analysis.py does) never pays for
    # matplotlib or forces its Agg backend unless a plot is actually requested.
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    os.makedirs(output_dir, exist_ok=True)
    data = dependence_plot_data(X_physical, shap_physical)

    paths = {}
    for name, values in data.items():
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.scatter(values["feature_value"], values["shap_value"], s=10, alpha=0.6)
        ax.axhline(0, color="grey", linewidth=0.5)
        ax.set_xlabel(name)
        ax.set_ylabel("SHAP value")
        ax.set_title(f"Tree SHAP dependence: {name}")
        path = os.path.join(output_dir, f"dependence_{name}.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        paths[name] = path
    return paths


def run_shap_analysis(
    model: xgb.XGBRegressor,
    X: np.ndarray,
    embedding_dim: int = EMBEDDING_DIM,
    physical_dim: int = PHYSICAL_DIM,
    top_k: int = DEFAULT_TOP_K,
    dependence_plot_dir: str | None = None,
) -> dict:
    """Run the full Tree SHAP analysis bundle against issue #14's acceptance criteria.

    Returns `top_embedding_features` (top-`top_k`, by mean |SHAP|),
    `physical_feature_shap` (all 4 named physical features), the aggregate
    `physical_contribution_ratio`, the raw `shap_values` matrix (for an
    `.npz` export), and `dependence_plot_paths` (empty unless
    `dependence_plot_dir` is given, since rendering plots is the only step
    here with a side effect - tests exercising the rest of this bundle don't
    need to write files).
    """
    shap_values = compute_tree_shap(model, X)
    groups = split_shap_by_group(shap_values, embedding_dim, physical_dim)

    dependence_plot_paths = {}
    if dependence_plot_dir is not None:
        X_physical = X[:, embedding_dim:]
        dependence_plot_paths = save_dependence_plots(
            X_physical, groups["physical"], dependence_plot_dir
        )

    return {
        "top_embedding_features": top_k_embedding_features(
            groups["embedding"], k=top_k
        ),
        "physical_feature_shap": physical_feature_shap_summary(groups["physical"]),
        "physical_contribution_ratio": aggregate_physical_contribution_ratio(
            shap_values, embedding_dim
        ),
        "shap_values": shap_values,
        "dependence_plot_paths": dependence_plot_paths,
    }


## 5. Confirm GPU is available

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected - Runtime > Change runtime type > GPU")

## 6. Load the Testset

In [ ]:
import pandas as pd

meta_df = pd.read_csv("test_metadata.csv")
sequences = meta_df["sequence"].tolist()
print(f"Loaded testset: {len(sequences)} samples.")

## 7. Load the production Model B predictor

In [ ]:
from model_b_wrapper import Model_B_Predictor

predictor = Model_B_Predictor(
    nt_model_dir="./NT_sacas9_fintuned_model", xgb_model_path="hybrid_xgb_model.json"
)

## 8. Build the full 8,964-feature Testset input matrix

`extract_nt_embeddings()` runs its whole input list through the NT transformer in one forward pass - unbounded memory, the same shape as the OOM issue #11 hit in `Model_A_Predictor.predict()`. `dispatch_in_batches` (from `ism_sweep.py`) bounds this the same way; lower `PREDICT_BATCH_SIZE` further if you still hit a CUDA OOM. `compute_physical_features()` is already internally batched (RNAfold/RNAduplex over stdin) and runs on CPU regardless.

In [ ]:
import time

import numpy as np
from ism_sweep import DEFAULT_PREDICT_BATCH_SIZE, dispatch_in_batches

PREDICT_BATCH_SIZE = (
    DEFAULT_PREDICT_BATCH_SIZE  # lower (e.g. 32) if you still hit CUDA OOM
)


def build_features(sequences):
    embeddings = dispatch_in_batches(
        predictor.extract_nt_embeddings, sequences, PREDICT_BATCH_SIZE, stack=np.vstack
    )
    physical = predictor.compute_physical_features(sequences)
    return np.hstack([embeddings, physical])


t0 = time.time()
X_test = build_features(sequences)
elapsed = time.time() - t0
assert X_test.shape == (len(sequences), 8964), (
    f"Expected ({len(sequences)}, 8964), got {X_test.shape}"
)
print(f"Feature extraction completed in {elapsed:.1f}s. X_test shape: {X_test.shape}")

## 9. Run Tree SHAP + physical-feature dependence plots

In [ ]:
from shap_analysis import run_shap_analysis

DEPENDENCE_PLOT_DIR = "shap_dependence_plots"

t0 = time.time()
result = run_shap_analysis(
    predictor.xgb_model, X_test, dependence_plot_dir=DEPENDENCE_PLOT_DIR
)
elapsed = time.time() - t0
print(f"Tree SHAP completed in {elapsed:.1f}s")

print("\n=== Top-20 Sequence-Embedding Features (mean |SHAP|) ===")
for entry in result["top_embedding_features"]:
    print(f" - feature[{entry['feature_index']}]: {entry['mean_abs_shap']:.6f}")
print("\n=== Physical Feature SHAP (mean |SHAP|) ===")
for entry in result["physical_feature_shap"]:
    print(f" - {entry['feature_name']}: {entry['mean_abs_shap']:.6f}")
print(
    f"\nAggregate physical SHAP contribution ratio: {result['physical_contribution_ratio']:.6f}"
)

## 10. Preview the dependence plots

In [ ]:
from IPython.display import Image, display

for name, path in result["dependence_plot_paths"].items():
    print(name)
    display(Image(filename=path))

## 11. Save and download the results

In [ ]:
import json
import shutil

np.savez("shap_values.npz", shap_values=result["shap_values"])

summary = {
    "top_embedding_features": result["top_embedding_features"],
    "physical_feature_shap": result["physical_feature_shap"],
    "physical_contribution_ratio": result["physical_contribution_ratio"],
    "dependence_plot_paths": result["dependence_plot_paths"],
}
with open("shap_results.json", "w") as f:
    json.dump(summary, f, indent=2)

# Also keep copies on Drive in case the runtime disconnects before you download.
np.savez(os.path.join(DRIVE_DIR, "shap_values.npz"), shap_values=result["shap_values"])
with open(os.path.join(DRIVE_DIR, "shap_results.json"), "w") as f:
    json.dump(summary, f, indent=2)
shutil.copytree(
    DEPENDENCE_PLOT_DIR,
    os.path.join(DRIVE_DIR, DEPENDENCE_PLOT_DIR),
    dirs_exist_ok=True,
)

from google.colab import files

files.download("shap_results.json")
files.download("shap_values.npz")
for path in result["dependence_plot_paths"].values():
    files.download(path)

Copy the downloaded `shap_results.json`, `shap_values.npz`, and `shap_dependence_plots/` into your local `xai_test/` directory - they have the same schema `step4_shap.py` produces locally.